In [ ]:
# --- IMPORTS ---

import os
import csv
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


# --- ENVIRONMENT AND MODEL SETUP ---

load_dotenv(find_dotenv())

openai_key = os.getenv("OPENAI_API_KEY")
assert openai_key, "OPENAI_API_KEY not found in .env"

openai_model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

cloud_llm = ChatOpenAI(
    model=openai_model,
    api_key=openai_key,
    temperature=0,
)

print(f"Using Cloud Model via OpenAI: {openai_model}")

# --- LOAD WEB SCANNER FINDINGS ---

import json
from pathlib import Path

RESULTS_PATH = Path("web_scan_results.json")

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        "web_scan_results.json not found. Run run_web_scanners.py first."
    )

with open(RESULTS_PATH, "r", encoding="utf-8") as file:
    scanner_results = json.load(file)

WEB_FINDINGS = [
    json.dumps(result, indent=2)
    for result in scanner_results
]

print(f"Loaded {len(WEB_FINDINGS)} scanner results.")

for finding in WEB_FINDINGS:
    print(finding)
    print("-" * 60)

Using Cloud Model via OpenAI: gpt-4o-mini
Loaded 3 scanner results.
{
  "scanner": "HTTP Security Header Scanner",
  "target": "http://127.0.0.1:3000/",
  "http_status": 200,
  "findings": [
    {
      "header": "Content-Security-Policy",
      "status": "MISSING",
      "observed_value": null,
      "finding": "Content-Security-Policy was not present in the response."
    },
    {
      "header": "X-Frame-Options",
      "status": "PRESENT",
      "observed_value": "SAMEORIGIN",
      "finding": "X-Frame-Options was present."
    },
    {
      "header": "X-Content-Type-Options",
      "status": "PRESENT",
      "observed_value": "nosniff",
      "finding": "X-Content-Type-Options was present."
    },
    {
      "header": "Referrer-Policy",
      "status": "MISSING",
      "observed_value": null,
      "finding": "Referrer-Policy was not present in the response."
    },
    {
      "header": "Permissions-Policy",
      "status": "MISSING",
      "observed_value": null,
      "findin

In [ ]:
# --- STAGE 1: Ungrounded Chain (No KB Retrieval) ---

report_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a web-application security assistant. Analyze the findings "
        "using your existing knowledge only. For each finding, provide a "
        "summary, potential severity, relevant CVE/CWE reference if known, "
        "and recommended remediation. Do not assume that every observation "
        "represents an exploitable vulnerability. Clearly identify uncertainty."
    ),
    ("user", "Findings:\n{findings}"),
])

stage1_chain = report_prompt | cloud_llm | StrOutputParser()


# --- GENERATE STAGE 1 REPORT ---

stage1_out = stage1_chain.invoke({
    "findings": "\n".join(
        f"- {finding}" for finding in WEB_FINDINGS
    )
})

print("=== STAGE 1 REPORT (UNGROUNDED — NO KB) ===\n")
print(stage1_out)


# --- SAVE STAGE 1 RESULTS ---

stage1_findings = [
    "HTTP Security Headers",
    "HTTP Method Enumeration",
    "Cookie Security"
]

with open(RESULTS_FILE, "a", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)

    for finding in stage1_findings:
        writer.writerow([
            finding,
            "UNGROUNDED",
            "",
            ""
        ])

print(f"Stage 1 results saved to {RESULTS_FILE}.")

Loaded 1 KB sections from web_security_kb.md.
=== STAGE 1 REPORT (UNGROUNDED — NO KB) ===

### Finding 1: Missing Content-Security-Policy Header
- **Summary**: The response from the server does not include a Content-Security-Policy (CSP) header, which is crucial for mitigating cross-site scripting (XSS) and other code injection attacks.
- **Potential Severity**: Medium to High, depending on the application's exposure to XSS vulnerabilities.
- **Relevant CVE/CWE Reference**: CWE-693: Protection Mechanism Failure.
- **Recommended Remediation**: Implement a Content-Security-Policy header that specifies allowed sources for scripts, styles, and other resources. For example: `Content-Security-Policy: default-src 'self'; script-src 'self';`.

### Finding 2: X-Frame-Options Header Present
- **Summary**: The X-Frame-Options header is present and set to "SAMEORIGIN", which helps prevent clickjacking attacks by allowing the page to be displayed in a frame only if the origin matches.
- **Potential

In [22]:
# --- KNOWLEDGE BASE SETUP ---

kb_path = Path("web_security_kb.md")

if not kb_path.exists():
    raise FileNotFoundError(
        f"{kb_path} not found. Make sure your KB is in the notebook directory."
    )

kb_text = kb_path.read_text(encoding="utf-8")

parts = kb_text.split("\n## ")

retrieved_blocks = [
    b.strip() for b in parts[1:] if b.strip()
] if len(parts) > 1 else [kb_text.strip()]

web_grounded_context = "\n\n---\n\n".join(retrieved_blocks)

print(
    f"Loaded {len(retrieved_blocks)} KB sections from {kb_path}."
)

# --- STAGE 2: Grounded Chain (With KB Context) ---

grounded_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a defensive web-application security analyst. "
        "Analyze each provided scanner observation using ONLY the "
        "provided Knowledge Base Context.\n\n"

        "For each observation:\n"
        "1. Identify the security issue or observation.\n"
        "2. Assign a classification: GROUNDED or GAP. If GROUNDED, provide the KB reference.\n"
        "3. Explain whether the KB supports the analysis.\n"
        "4. Provide severity and remediation only when supported by the KB.\n\n"

        "GROUNDED means the KB contains relevant information supporting "
        "the analysis.\n"
        "GAP means the KB does not contain sufficient information to "
        "support the analysis.\n\n"

        "Do not invent CVE/CWE references, severity ratings, or remediation. "
        "Do not assume that every scanner observation represents a "
        "vulnerability. Clearly identify observations where no vulnerability "
        "was established."
    ),
    (
        "user",
        "Scanner Findings:\n{findings}\n\n"
        "Knowledge Base Context:\n{context}"
    ),
])

stage2_chain = grounded_prompt | cloud_llm | StrOutputParser()


# --- GENERATE STAGE 2 REPORT ---

import time

start_time = time.perf_counter()

stage2_out = stage2_chain.invoke({
    "findings": "\n".join(WEB_FINDINGS),
    "context": web_grounded_context
})

phase2_time = time.perf_counter() - start_time

print("=== STAGE 2 REPORT (GROUNDED — WITH KB) ===\n")
print(stage2_out)
print(f"\nPhase 2 execution time: {phase2_time:.2f} seconds")

import csv
import re
from pathlib import Path

grounded_count = len(re.findall(
    r"\*\*Classification\*\*:\s*GROUNDED",
    stage2_out,
    re.IGNORECASE
))

gap_count = len(re.findall(
    r"\*\*Classification\*\*:\s*GAP",
    stage2_out,
    re.IGNORECASE
))

RESULTS_FILE = "results.csv"

# Append Phase 2 results
with open(RESULTS_FILE, "a", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow([
        "Phase 2",
        "v2",
        grounded_count,
        gap_count,
        round(phase2_time, 2)
    ])

print(f"Phase 2: {grounded_count} GROUNDED, {gap_count} GAP")
print(f"Execution time: {phase2_time:.2f} seconds")
print(f"Results saved to {RESULTS_FILE}.")

Loaded 1 KB sections from web_security_kb.md.
=== STAGE 2 REPORT (GROUNDED — WITH KB) ===

### Observation 1: Missing Content-Security-Policy Header
1. **Security Issue**: The Content-Security-Policy (CSP) header is missing in the response.
2. **Classification**: GROUNDED. Reference: Missing Content-Security-Policy (CSP) header.
3. **KB Support**: The KB explicitly states that the absence of a CSP header is a security misconfiguration and can lead to vulnerabilities such as Cross-Site Scripting (XSS).
4. **Severity**: Not specified in the KB. **Remediation**: Define a restrictive policy such as `Content-Security-Policy: default-src 'self'; object-src 'none'; frame-ancestors 'none'` and tighten it per page.

### Observation 2: X-Frame-Options Header Present
1. **Security Issue**: The X-Frame-Options header is present with a value of SAMEORIGIN.
2. **Classification**: GROUNDED. Reference: Missing X-Frame-Options header (clickjacking).
3. **KB Support**: The KB indicates that having the X